# DM_TF — Cuadernillo Python Único
**Tipología agroclimática y producción agrícola en el Perú (2020–2025)**  
Universidad del Pacífico — Data Mining 2026-I


## Mapa CRISP-DM → secciones

| Fase CRISP-DM | Secciones del cuadernillo |
|---------------|---------------------------|
| 1. Comprensión del negocio | Portada, objetivos |
| 2. Comprensión de datos | MIDAGRI, NASA POWER |
| 3. Preparación | Merge + Pareto-80 + mapping v2 |
| 4. Modelado | Clustering, PCA, correlaciones |
| 5. Evaluación | Robustez (ARI, Hopkins, semillas) |
| 6. Despliegue | Reproducibilidad, limitaciones |

**Entregables generados:** `dataset_integrado.csv`, figuras en `OUTPUTS/figures/`, métricas de clustering.

In [1]:
# =============================================================================
# Celda 2 — Setup Colab / local
# =============================================================================
# REQUIERE RED: clone + pip (y NASA API solo si REDESCARGAR_NASA=True)

import sys
import subprocess
from pathlib import Path

MODO_DEMO = True           # True → CSV cacheados (recomendado en Colab / exposición)
REPROCESAR_MIDAGRI = False # True → necesita Excel en BDS/YYYY/MES.xlsx
REDESCARGAR_NASA = False   # True → ~14 llamadas NASA POWER (~10-15 min)
UMBRAL_PARETO = 0.80
RANDOM_STATE = 42

# --- Detectar raíz del repo ---
_cwd = Path.cwd().resolve()
if (_cwd / "SCRIPTS" / "pipeline_integrado.py").exists():
    ROOT = _cwd
elif (_cwd.parent / "SCRIPTS" / "pipeline_integrado.py").exists():
    ROOT = _cwd.parent
elif _cwd.name == "ENTREGAS" and (_cwd.parent / "SCRIPTS" / "pipeline_integrado.py").exists():
    ROOT = _cwd.parent
else:
    ROOT = Path("/content/DM_TF")
    REPO_URL = "https://github.com/JohaoM-ML/DM_TF.git"
    if not ROOT.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(ROOT)], check=True)

SCRIPTS = ROOT / "SCRIPTS"
if str(SCRIPTS) not in sys.path:
    sys.path.insert(0, str(SCRIPTS))

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements.txt")],
    check=True,
)

from paths import (
    MAPPING_ACTIVO, OUTPUTS, OUTPUTS_FIGURES, OUTPUTS_ROBUSTEZ,
    MIDAGRI_LARGO, NASA_CLIMA, DATASET_INTEGRADO, DATASET_REGIONAL,
)

OUTPUTS.mkdir(parents=True, exist_ok=True)
OUTPUTS_FIGURES.mkdir(parents=True, exist_ok=True)
OUTPUTS_ROBUSTEZ.mkdir(parents=True, exist_ok=True)

insumos = {
    "midagri_largo.csv": MIDAGRI_LARGO,
    "nasa_2020_2025.csv": NASA_CLIMA,
    "mapping_v2": MAPPING_ACTIVO,
}
for nombre, ruta in insumos.items():
    print(f"  [{'OK' if ruta.exists() else 'FALTA'}] {nombre}: {ruta}")

if MODO_DEMO and not all(p.exists() for p in insumos.values()):
    raise FileNotFoundError(
        "MODO_DEMO=True pero faltan CSV. Clone el repo completo o suba OUTPUTS/ y BDS/mapping/."
    )

import pandas as pd
import matplotlib.pyplot as plt

from pipeline_integrado import ejecutar_desde_csvs
from clustering import clustering_perfiles, CLIMA_CORE
from robustez import pareto_sensibilidad, estabilidad_semillas, hopkins_statistic, ari_v1_v2

pd.set_option("display.max_columns", 25)
plt.rcParams["figure.dpi"] = 120

print(f"\nROOT={ROOT}")
print(f"MODO_DEMO={MODO_DEMO} | REDESCARGAR_NASA={REDESCARGAR_NASA}")
print("Setup completado")

CalledProcessError: Command '['c:\\Users\\USER\\anaconda3\\python.exe', '-m', 'pip', 'install', '-q', '-r', 'C:\\Users\\USER\\OneDrive\\Escritorio\\Github\\DM_TF\\requirements.txt']' returned non-zero exit status 1.

## CRISP-DM 1 — Comprensión del negocio

**Problema:** caracterizar perfiles agroclimáticos región×cultivo en seis regiones peruanas (2020–2025).

**Objetivos:**
1. Integrar producción MIDAGRI (C-18) con clima NASA POWER vía mapping documentado.
2. Construir dataset maestro con filtro Pareto-80 regional.
3. Comparar tareas de Data Mining: asociación, clustering, reducción dimensional, robustez.
4. Entregar tipología descriptiva (K clusters) sin afirmar causalidad.

**Alcance:** 6 regiones, 14 distritos NASA, 33 perfiles Pareto-80, 2.376 observaciones mensuales.

In [ ]:
# Verificación de insumos
checks = {
    "midagri_largo": MIDAGRI_LARGO.exists(),
    "nasa_clima": NASA_CLIMA.exists(),
    "mapping_v2": MAPPING_ACTIVO.exists(),
    "dataset_integrado": DATASET_INTEGRADO.exists(),
}
for k, ok in checks.items():
    print(f"[{'OK' if ok else 'FALTA'}] {k}")
if not all(checks.values()) and MODO_DEMO:
    raise FileNotFoundError("Faltan insumos para MODO_DEMO. Ejecute notebooks 01-02 o use el repo completo.")

## CRISP-DM 2 — Comprensión de datos (MIDAGRI)

- Fuente: cuadro **C-18** MIDAGRI (producción **acumulada** mensual).
- **Producción mensual = diff(acumulado)**; negativos del diff → NaN (revisiones retroactivas).
- **NaN en meses faltantes:** no imputamos (p. ej. Mayo-2021, Mar-2022 quedan explícitos).
- Recuperación Dic-2020 desde archivo 2021 documentada en `01_midagri_pipeline.ipynb`.

In [ ]:
# Carga MIDAGRI (rama demo: CSV cacheado)
df_midagri = pd.read_csv(MIDAGRI_LARGO)
print(f"MIDAGRI: {len(df_midagri):,} filas | regiones: {df_midagri['region'].nunique()}")
print(f"Rango años: {df_midagri['anio'].min()}–{df_midagri['anio'].max()}")
print(f"NaN produccion_mensual: {df_midagri['produccion_mensual'].isna().sum()}")

## CRISP-DM 2 — Comprensión de datos (NASA POWER)

- 14 distritos representativos, 12 variables climáticas, 2020–2025.
- Sentinel `-999` → NaN; descarte mes 13 si aplica.
- 🌐 **REQUIERE RED** solo si `REDESCARGAR_NASA=True` (ver `02_nasa_pipeline.ipynb`).

In [ ]:
df_nasa = pd.read_csv(NASA_CLIMA)
print(f"NASA: {len(df_nasa):,} filas | distritos: {df_nasa['distrito'].nunique()}")
print(f"Variables: {[c for c in df_nasa.columns if c not in ('distrito','anio','mes_num','region')][:5]}...")

## CRISP-DM 3 — Preparación: merge + Pareto-80

- **Mapping v2 canónico:** `mapping_cultivo_distrito_v2_pipeline.csv` (cultivo→piso→distrito).
- **Pareto-80 corregido:** incluir cultivos hasta alcanzar **≥80%** acumulado regional (no ≤80%).
- **Agregado regional:** `sum(min_count=1)` — meses 100% NaN no se convierten en 0.
- **Clima compartido por piso:** cultivos del mismo (región, piso) tienen la misma serie climática.

In [ ]:
result = ejecutar_desde_csvs(OUTPUTS, MAPPING_ACTIVO, umbral_pareto=UMBRAL_PARETO)
df_integrado = result["dfs"]["dataset_integrado"]

print("=== Pareto-80 ===")
for line in result["pareto_reporte"]:
    print(line)
print("\n=== Validación ===")
for k, v in result["stats"].items():
    print(f"  {k}: {v}")

n_combos = df_integrado.groupby(["region", "cultivo"]).ngroups
print(f"\nDataset integrado: {df_integrado.shape[0]:,} × {df_integrado.shape[1]} | combos: {n_combos}")

In [ ]:
# Evidencia: clima compartido dentro del mismo piso (Ica costa)
ica = df_integrado[(df_integrado["region"] == "Ica") & (df_integrado["anio"] == 2020) & (df_integrado["numero_mes"] == 1)]
cols = ["cultivo", "distrito", "temp_promedio", "produccion_ton"]
print(ica[cols].drop_duplicates().head(8).to_string(index=False))
print("\n→ Espárrago y uva en Ica costa comparten temp_promedio idéntica por diseño.")

## CRISP-DM 2/3 — EDA regional

Análisis **descriptivo** de producción por región/piso. No implica causalidad climática.

In [ ]:
import seaborn as sns

df_reg = pd.read_csv(DATASET_REGIONAL)
fig, ax = plt.subplots(figsize=(10, 4))
prod_anual = df_reg.groupby(["region", "anio"])["produccion_piso_ton"].sum().reset_index()
for region in sorted(prod_anual["region"].unique()):
    sub = prod_anual[prod_anual["region"] == region]
    ax.plot(sub["anio"], sub["produccion_piso_ton"], marker="o", label=region)
ax.set_title("Producción anual agregada por región (ton)")
ax.set_xlabel("Año"); ax.legend(bbox_to_anchor=(1.02, 1)); plt.tight_layout()
out = OUTPUTS_FIGURES / "eda_regional_produccion_anual.png"
fig.savefig(out, dpi=120); plt.show()
print(f"Guardado: {out}")

## CRISP-DM 2/3 — EDA por cultivo (tarea DM: **asociación**)

Correlaciones **Pearson exploratorias** clima–producción por (región, cultivo).

**Limitaciones:** sin desestacionalización, sin corrección Benjamini–Hochberg, volumen en ton (no t/ha).

In [ ]:
from scipy import stats

df = pd.read_csv(DATASET_INTEGRADO)
rows = []
for (region, cultivo), g in df.groupby(["region", "cultivo"]):
    for var in CLIMA_CORE:
        sub = g[[var, "produccion_ton"]].dropna()
        if len(sub) < 8:
            continue
        r, p = stats.pearsonr(sub[var], sub["produccion_ton"])
        rows.append({"region": region, "cultivo": cultivo, "variable_clima": var, "n": len(sub), "r": r, "p_valor": p})

corr_df = pd.DataFrame(rows)
corr_df["abs_r"] = corr_df["r"].abs()
top5 = corr_df.nlargest(5, "abs_r")
print("=== Top 5 |r| (exploratorio, sin BH) ===")
print(top5[["region", "cultivo", "variable_clima", "n", "r", "p_valor"]].to_string(index=False))
corr_df.to_csv(OUTPUTS / "eda_correlaciones_por_cultivo.csv", index=False, encoding="utf-8-sig")

## CRISP-DM 4 — Modelado: unidad de análisis

- **Perfil** = agregado región×cultivo (`construir_perfiles`).
- **33 perfiles** Pareto-80 — clustering sobre perfiles, no sobre 2.376 filas mensuales.
- **>3 tareas DM en este bloque:** clustering (KMeans/jerárquico/DBSCAN), reducción dimensional (PCA), detección de ruido (DBSCAN).

In [ ]:
clust = clustering_perfiles(
    pd.read_csv(DATASET_INTEGRADO),
    OUTPUTS,
    OUTPUTS_FIGURES,
    random_state=RANDOM_STATE,
)
print(f"K={clust['k']} | Silhouette={clust['silhouette']:.4f} | {clust['motivo_k']}")
print(f"Perfiles: {clust['n_perfiles']}")
from IPython.display import display
display(clust["metricas"])

In [ ]:
from IPython.display import Image, display
fig_k = OUTPUTS_FIGURES / "seleccion_k_perfiles.png"
if fig_k.exists():
    display(Image(filename=str(fig_k)))
print("Selección K=6: compromiso Silhouette + Davies-Bouldin (ver clustering.py::elegir_k)")

In [ ]:
# DBSCAN: alto % ruido → no es método principal
db_row = clust["metricas"][clust["metricas"]["metodo"] == "DBSCAN"]
if not db_row.empty:
    print(db_row[["metodo", "silhouette", "pct_ruido", "motivo_k"]].to_string(index=False))
    print("\n→ Silhouette DBSCAN se calcula EXCLUYENDO etiqueta -1 (ruido).")

In [ ]:
# PCA complementario (reducción dimensional sobre filas mensuales, solo clima)
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

df = pd.read_csv(DATASET_INTEGRADO)
perfiles = pd.read_csv(OUTPUTS / "clustering_perfiles.csv")
X = StandardScaler().fit_transform(df[CLIMA_CORE].fillna(df[CLIMA_CORE].median()))
pca = PCA(n_components=2, random_state=RANDOM_STATE)
coords = pca.fit_transform(X)
df_pca = df[["region", "cultivo"]].copy()
df_pca["PC1"], df_pca["PC2"] = coords[:, 0], coords[:, 1]
df_pca = df_pca.merge(perfiles[["region", "cultivo", "cluster_kmeans"]], on=["region", "cultivo"], how="left")

fig, ax = plt.subplots(figsize=(8, 5))
for c in sorted(df_pca["cluster_kmeans"].dropna().unique()):
    sub = df_pca[df_pca["cluster_kmeans"] == c]
    ax.scatter(sub["PC1"], sub["PC2"], alpha=0.4, s=15, label=f"C{int(c)}")
ax.set_title(f"PCA clima (var PC1+PC2={pca.explained_variance_ratio_.sum():.1%})")
ax.legend(); plt.tight_layout(); plt.show()

## CRISP-DM 5 — Evaluación: robustez

Sensibilidad Pareto 70/80/90, estabilidad de semillas, Hopkins, ARI mapping v1↔v2.

In [ ]:
print("=== Sensibilidad Pareto ===")
print(pareto_sensibilidad(OUTPUTS, MAPPING_ACTIVO).to_string(index=False))

print("\n=== Estabilidad semillas ===")
print(estabilidad_semillas(pd.read_csv(DATASET_INTEGRADO)).to_string(index=False))

from clustering import construir_perfiles
from sklearn.preprocessing import StandardScaler
df_p = pd.read_csv(DATASET_INTEGRADO)
prof, feats = construir_perfiles(df_p)
X_h = StandardScaler().fit_transform(prof[feats])
h = hopkins_statistic(X_h)
print(f"\nHopkins: {h:.4f}  (≈0.5 = estructura moderada)")

ari_path = OUTPUTS_ROBUSTEZ / "ari_v1_v2.csv"
if ari_path.exists():
    print("\n=== ARI v1 vs v2 (pre-calculado) ===")
    print(pd.read_csv(ari_path).to_string(index=False))
else:
    print("\n[INFO] Ejecute 'make robustez' para generar ari_v1_v2.csv")

In [ ]:
fig_hm = OUTPUTS_FIGURES / "heatmap_perfiles_kmeans.png"
if fig_hm.exists():
    display(Image(filename=str(fig_hm)))

perfiles = pd.read_csv(OUTPUTS / "clustering_perfiles.csv")
print("\nComposición por cluster:")
print(perfiles.groupby("cluster_kmeans").agg(n=("cultivo", "count"), regiones=("region", "nunique")))

## CRISP-DM 5 — Comparativa de tareas de Data Mining

| Tarea DM | Método | Métrica principal | Resultado |
|----------|--------|-------------------|-----------|
| Asociación | Pearson por cultivo | r, p | Exploratorio; ver CSV |
| Clustering | KMeans K=6 | Silhouette ≈ 0,51 | Tipología 33 perfiles |
| Clustering alt. | Jerárquico Ward | Silhouette | Coincide con KMeans |
| Detección densidad | DBSCAN | % ruido | ~64% ruido → descartado |
| Reducción dim. | PCA (clima) | Varianza PC1+PC2 | Visualización mensual |
| Validación estructura | Hopkins | H | ≈ 0,54 (moderado) |

**Recomendación:** KMeans K=6 por balance métrica/interpretación.

## CRISP-DM 6 — Despliegue / reproducibilidad

```bash
pip install -r requirements.txt
python SCRIPTS/pipeline_integrado.py
make cluster
make robustez
pytest tests/ -v
```

- **Modo demo (exposición):** `MODO_DEMO=True`, sin red NASA.
- **Reproducción completa:** notebooks `01_midagri_pipeline.ipynb` y `02_nasa_pipeline.ipynb`.

In [ ]:
stats = result["stats"]
assert stats.get("max_diff_a_b", 1) < 1e-5 or "max_diff_a_b" not in stats
assert df_integrado.groupby(["region", "cultivo"]).ngroups == 33
assert 2 <= clust["k"] <= 7
print("Cuadernillo reproducible: shapes, Pareto-33, K en rango OK")

## Conclusiones

- **33 perfiles** región×cultivo (Pareto-80, mapping v2) sobre **2.376** filas mensuales.
- **K=6** clusters con Silhouette ≈ 0,51 — estructura **moderada**, no perfecta.
- Tipologías interpretables: costa Ica, selva Junín/SM, altiplano Puno, outlier caña La Libertad.
- Sequía 2022 visible en series Puno (EDA regional).
- Resultado **sensible** al mapping (ARI v1↔v2 ≈ 0,87; K cambió 7→6).

## Limitaciones metodológicas (obligatorio)

1. **Clima constante por piso/distrito** — no discrimina sensibilidad intra-piso entre cultivos.
2. **Producción en toneladas (volumen), no t/ha** — sin área sembrada no se aísla productividad.
3. **No causalidad / no predicción** — tipología exploratoria para priorización de EDA.
4. **n ≈ 33 perfiles** — clusters pequeños; Hopkins ≈ 0,5.
5. **Correlaciones sin BH** ni desestacionalización.
6. **K sensible** al umbral Pareto y al mapping manual.

Ver también: `ENTREGAS/LIMITACIONES.md`, `DOCUMENTACIÓN/guion_defensa.md`.

## Implicancias éticas (rúbrica informe)

- Datos públicos agregados (MIDAGRI, NASA) — sin datos personales de productores.
- **Sesgo Pareto-80:** marginaliza cultivos de seguridad alimentaria (quinua, oca) fuera del análisis.
- **Mapping manual:** errores de asignación pueden atribuir clima incorrecto a comunidades reales.
- Evitar lenguaje de "vulnerabilidad" o "impacto" sin evidencia causal — riesgo de estigmatizar regiones.

## Trabajos futuros (rúbrica informe)

- Integrar área sembrada MIDAGRI → rendimiento (t/ha).
- Variables derivadas: VPD, grados-día, SPI desde series existentes.
- Covariables ENSO (ONI) como contexto macro, no causal.
- Nuevas tareas DM: reglas de asociación, detección de anomalías, forecasting con validación temporal.
- Validación con agrónomos regionales; cuadernillo Colab ya preparado en este archivo.

## Apéndice — mapa notebooks → celdas

| Notebook | Celdas equivalentes |
|----------|---------------------|
| 01_midagri | 5–6 |
| 02_nasa | 7–8 |
| 03_build | 9–11 |
| 04_eda_regional | 12–13 |
| 05_eda_cultivo | 14–15 |
| 06_clustering | 16–21 |
| robustez | 22–23 |

Los notebooks `SCRIPTS/notebooks/01–06` permanecen como referencia modular; **este archivo es el entregable único** para la rúbrica.

In [ ]:
# Metadatos de entrega
import subprocess
try:
    commit = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], cwd=ROOT, text=True).strip()
except Exception:
    commit = "N/A"
print(f"Proyecto: DM_TF | mapping: v2 | commit: {commit}")
print(f"Cuadernillo: ENTREGAS/DM_TF_cuadernillo_completo.ipynb")